# 🏦 Interactive Blueprint: Análise do Setor Financeiro (B3)

Este notebook é a Central de Inteligência para o setor financeiro. 

### 🎮 Como usar:
1. Execute as células abaixo.
2. Use o menu à esquerda para **selecionar os ativos** que deseja comparar.
3. Todos os 4 dashboards (Performance, Risco, Eficiência e Correlação) serão atualizados automaticamente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact, interactive_output, HBox, VBox
from IPython.display import display, clear_output

# Configurações estéticas
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Caminho para o setor financeiro (Parquet Particionado)
caminho_setor = '../data/processed/setores/Financial_Services.parquet'
df = pd.read_parquet(caminho_setor)
df['date'] = pd.to_datetime(df['date'])

tickers_disponiveis = sorted(df['ticker'].unique())
print(f"✅ {len(tickers_disponiveis)} ativos financeiros carregados.")

### 🛠️ Configuração do Seletor de Ativos

In [ ]:
# Filtramos defaults que existem de fato nos dados para evitar erro de inicialização
defaults_desejados = ['BBAS3.SA', 'BBDC4.SA', 'SANB11.SA', 'B3SA3.SA', 'ABCB4.SA']
defaults_reais = [t for t in defaults_desejados if t in tickers_disponiveis]

# Se nenhum do top 5 existir, pega os 5 primeiros da lista
if not defaults_reais:
    defaults_reais = tickers_disponiveis[:5]

selector = widgets.SelectMultiple(
    options=tickers_disponiveis,
    value=defaults_reais,
    description='Ativos',
    disabled=False,
    layout=widgets.Layout(width='300px', height='200px')
)

print("Selecione os ativos (segure CTRL para múltiplos):")
display(selector)

## 📊 Painel de Análise Dinâmico
Execute a célula abaixo para ativar os dashboards.

In [ ]:
out = widgets.Output()

def update_dashboards(selected_tickers):
    with out:
        clear_output(wait=True)
        
        if not selected_tickers:
            print("⚠️ Por favor, selecione ao menos um ativo.")
            return
            
        df_comp = df[df['ticker'].isin(selected_tickers)].copy()
        
        # 1️⃣ PERFORMANCE
        fig1 = px.line(df_comp, x='date', y='retorno_acumulado', color='ticker', 
                      title='📈 Performance Acumulada: Histórico',
                      labels={'retorno_acumulado': 'Crescimento (%)'})
        fig1.show()
        
        # 2️⃣ RISCO vs RETORNO (Sharpe)
        df_risco = df_comp.groupby('ticker').agg({
            'sharpe_ratio_21d': 'mean', 
            'volatilidade_252d': 'mean',
            'drawdown': 'min'
        }).reset_index()
        
        fig2 = px.scatter(df_risco, x='volatilidade_252d', y='sharpe_ratio_21d', 
                         size=np.abs(df_risco['drawdown']), text='ticker', color='ticker',
                         title='🎯 Risco (Vol) vs Retorno (Sharpe) | Tamanho = Max Drawdown',
                         labels={'volatilidade_252d': 'Volatilidade (Risco)', 'sharpe_ratio_21d': 'Sharpe Ratio'})
        fig2.show()
        
        # 3️⃣ EFICIÊNCIA (ROE & Payout)
        df_latest = df_comp.sort_values('date').groupby('ticker').tail(1)
        
        fig3 = px.bar(df_latest, x='ticker', y='roe', color='payout_ratio', 
                     title='💎 Rentabilidade (ROE) vs Taxa de Payout (Cores)',
                     labels={'roe': 'ROE', 'payout_ratio': 'Payout Ratio'})
        fig3.show()
        
        # 4️⃣ CORRELAÇÃO
        if len(selected_tickers) > 1:
            df_pivot = df_comp.pivot(index='date', columns='ticker', values='log_return').dropna()
            if not df_pivot.empty:
                corr = df_pivot.corr()
                plt.figure(figsize=(8, 6))
                sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0, fmt=".2f")
                plt.title('🔥 Matriz de Correlação entre Ativos Selecionados')
                plt.show()
            else:
                print("⚠️ Dados insuficientes para matriz de correlação (datas não batem).")

# Vínculo entre widget e função
def on_value_change(change):
    update_dashboards(change['new'])

selector.observe(on_value_change, names='value')

# Inicializa o dashboard com os valores padrão
update_dashboards(selector.value)
display(out)